# Wrong-way detection on a Colab GPU

Runs `pipeline.py` (RF-DETR + ByteTrack + `WrongWayDetector`) against
traffic video.

The code executes on a **remote Google machine**, which cannot see your
local disk — so the cells below fetch the code from GitHub and the video
from either the `supervision` sample set, Drive, or an upload.

**Pick a GPU runtime before running:** Runtime → Change runtime type →
T4 GPU. The first cell says plainly whether you got one. On CPU
everything still works, roughly an order of magnitude slower.

**Read the class-id table in the run cell's output before you read any
alert.** COCO has two numbering schemes that disagree about every
vehicle, and picking the wrong one means tracking bicycles and trains
while every bus and truck is silently discarded — which is exactly what
happened here for two runs. The table now shows both interpretations
where the model will not name its own classes; resolve it by looking at
what is actually on the road in the footage.

**Two rules that have each cost an hour:**

- After every push, re-run the git cell. A running kernel keeps executing
  the code already in memory, so a fixed bug reproduces identically.
- If anything behaves as though the old code is still running, restart
  the runtime and run from the top. Three minutes of re-running beats
  twenty minutes chasing a bug that was already fixed.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Switch the runtime to a GPU type, then rerun this cell.")

## 1. Install

Only two packages. The runtime already ships `torch` (CUDA build),
`opencv` and `numpy`; installing `requirements.txt` wholesale can replace
the CUDA torch with a CPU one and silently cost you the GPU.
`supervision` arrives as a dependency of `rfdetr`.

In [ ]:
!pip install -q rfdetr trackers

## 2. Get the code

Clones on the first run of a session, pulls on every run after that.
**Re-run this cell after every push** — it is what carries your local
edits across to the machine that actually executes them.

It also clears our modules from Python's import cache. Pulling changes
the files on disk, but a running kernel keeps executing the copies
already in memory, so a fixed bug reproduces identically and the fix
looks like it failed. That is the most common reason "I already fixed
that" turns out to be false in any Jupyter session.

If anything still behaves as though the old code is running, restart the
runtime and run from the top. That always works, and three minutes of
re-running beats twenty minutes chasing a bug that was already fixed.

In [ ]:
import os
import sys

REPO_URL = "https://github.com/Arielevi15/Crime_Traffic_Dedector.git"
REPO_DIR = "/content/Crime_Traffic_Dedector"

if not os.path.isdir(REPO_DIR):
    !git clone --quiet {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull --ff-only

# Drop our modules from Python's import cache. Without this, `git pull`
# updates the files on disk while the kernel keeps executing the copies
# already in memory -- a fixed bug reproduces identically and the fix
# looks like it failed. The next import below re-reads from disk.
#
# (IPython's %autoreload would also do this, but it imports `imp`, which
# Python 3.12 removed, so it cannot load on this runtime at all.)
for _name in [n for n in list(sys.modules) if n.startswith("road_crime")]:
    sys.modules.pop(_name, None)

!ls

### 2b. Prove the violation logic that just arrived is the one that will run

The cell above pulls files and drops our modules from the import cache.
Neither is a guarantee, and the failure they guard against is silent: the
kernel keeps executing code it already holds, so a fixed bug reproduces
exactly and the fix looks like it failed.

So check instead of assuming. This imports `wrong_way_detector`, prints
the file it actually resolved to, the commit the checkout is on, and the
thresholds now in force, then runs both synthetic suites. They need no
GPU, no video and no RF-DETR, so they cost seconds — and they are the
only evidence that the decision logic is sound *before* a frame is
decoded.

If the resolved path is not inside `/content/Crime_Traffic_Dedector`, or
a suite fails, stop here. Everything below would be describing code you
are not running.

In [ ]:
import os
import subprocess
import sys

import road_crime.wrong_way_detector as wrong_way
from road_crime.wrong_way_detector import DetectorConfig

# Normalise both sides before comparing. A raw `startswith` on paths
# disagrees over symlinks, trailing separators and drive-letter case, and
# a check that fails spuriously is worse than no check: it stops you for a
# reason that is not real, and you learn to ignore it.
_resolved = os.path.realpath(wrong_way.__file__)
_expected = os.path.realpath(REPO_DIR)

print("wrong_way_detector resolved to:")
print("   ", _resolved)
assert os.path.commonpath([_resolved, _expected]) == _expected, (
    "The module came from {0}, not from the checkout you just pulled "
    "({1}). Restart the runtime and run from the top.".format(_resolved, _expected)
)

print()
print("checkout is on:")
print("   ", subprocess.run(["git", "log", "-1", "--oneline"], cwd=_expected,
                            capture_output=True, text=True).stdout.strip())

# The thresholds that will actually decide every alert below. Printed
# because a config is easy to change in passing and hard to notice
# afterwards, and because "0 alerts" means something different at each of
# these values.
print()
print("thresholds in force:")
for _field in ("zone_size", "baseline_min_samples", "baseline_min_coherence",
               "opposite_cos_threshold", "violation_frames_required",
               "min_speed_fraction", "peer_min_tracks"):
    print("    {0:<26} {1}".format(_field, getattr(DetectorConfig(), _field)))


def run_suite(module):
    """Run one synthetic suite and show what it said.

    `cwd` is passed explicitly rather than inherited. `python -m tests.X`
    resolves `tests` from the working directory, so a subprocess started
    from anywhere else dies with `No module named tests` -- which is not a
    test failure at all, but looks exactly like one. That is what broke
    this cell the first time it ran on a fresh runtime.

    Output is captured and printed rather than left to `check=True`. A
    bare CalledProcessError names the command and hides the reason, so a
    genuinely failing test and a mislocated one produce the same opaque
    traceback and you cannot tell which you have.
    """
    result = subprocess.run(
        [sys.executable, "-m", module],
        cwd=_expected, capture_output=True, text=True,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if result.returncode != 0:
        print(result.stderr.rstrip())
        raise SystemExit(
            "{0} did not pass (exit {1}). Read the lines above: a FAIL names "
            "the test, anything else is an environment problem rather than "
            "the logic.".format(module, result.returncode)
        )


# Both suites, not just the wrong-way one: main stays green only if the
# other track's tests pass too (WORKPLAN rule 0.6).
print()
run_suite("tests.test_wrong_way")
run_suite("tests.test_stop_sign")

## 3. Get the video

Two options. Run **3a** for the very first run, and **3b** once you have
real footage -- they answer different questions.

### 3a. Sample footage (start here)

One line, no upload, no Drive. `supervision` ships it, and it is already
installed as a dependency of `rfdetr`.

Be clear about what this does and does not prove. It is **elevated
highway footage, not a forward-facing dashcam**, so it does not match the
scope assumption at the top of `CLAUDE.md`. Good for: confirming the
class ids, that RF-DETR loads, that ByteTrack holds ids across frames,
and that the chain runs end to end. Not good for: tuning any threshold in
`DetectorConfig`, or judging the wrong-way logic in the domain we
actually care about.

In [ ]:
from supervision.assets import VideoAssets, download_assets

VIDEO = download_assets(VideoAssets.VEHICLES)
print("Video ready:", VIDEO)

### 3b. Your own dashcam footage

Skip this on the first run. Once you have real clips, put them in a Drive
folder once and every future session sees them without another upload.
Mounting opens an auth prompt the first time. Running this cell
overwrites `VIDEO` from 3a.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Point this at your own clip. Not sure of the exact path? Find it:
#   !find /content/drive/MyDrive -iname "*.mp4" | head -20
# Note that Drive's "Shared with me" is a view, not a folder -- it is
# never mounted. Add a shortcut to My Drive, or copy the file there.
candidate = "/content/drive/MyDrive/dashcam/sample1.mp4"

# Check first, assign second. Assigning before the check would let a
# wrong path here silently replace a working VIDEO set by cell 3a, and
# the failure would then surface much later, in the run cell.
assert os.path.isfile(candidate), (
    "Not found: {0}\nRun the find command above to check the name.".format(candidate)
)
VIDEO = candidate
print("Video ready:", VIDEO)

### 3c. A clip from YouTube

For the one thing neither 3a nor 3b can supply: an actual wrong-way
driver. Ordinary footage contains none, which is why `evaluate.py`
measures sensitivity by replaying real trajectories backwards. An
injection is a good stand-in, but it is still a stand-in — nobody has yet
seen this module fire on a genuine violation.

News footage is where those events surface, and it arrives with three
problems.

**Trim to the dashcam segment first.** A news package is mostly studio,
b-roll and interviews, with a few seconds of dashcam inside it.
`DirectionBaseline` is global and survives every cut, so untrimmed it
learns "directions of travel" from a presenter's shoulders and then
judges the dashcam segment against them — a vehicle measured by a rule
learned somewhere else entirely. For a new URL, step through the frames
before choosing `START`: guessing from the runtime alone put an earlier
version of this cell squarely inside an interview.

**Ask for a real resolution.** YouTube's pre-merged mp4 stops at 360p, so
`format="mp4"` quietly gives you 640x360. A vehicle approaching head-on
is a handful of pixels until it is nearly on top of the camera, and the
module needs roughly 18 consecutive frames on one track id before it can
say anything. Take the video-only stream instead; the selector below
does, and the resolution print at the end is there to catch it when it
does not.

**Most of this footage is all-rights-reserved.** Keep the video on this
runtime and bring home only the `.jsonl` dump. The dump is all a
violation module ever sees anyway (principle 5), and it is what
`fixtures/` is for.

Like 3b, this overwrites `VIDEO`. Then change two things in the run cell
below — drop `limit_frames`, and lower `conf`, because an approaching
vehicle is motion-blurred and changing size fast:

```python
alerts = run(video=VIDEO, output="check.mp4", conf=0.35, dump_tracks=TRACKS)
```

**Expect silence on a clip this short, and read it correctly.** Five
seconds of near-empty freeway cannot put 30 heading samples
(`baseline_min_samples`) into any one zone, so no zone becomes trusted
and the detector never judges anybody — the same shortage measured on
`divided_highway_clean.jsonl`, where only 6 zones of 21 ever qualified.
That is the module declining to guess, which principle 3 asks of it, but
it means this clip tests perception and tracking rather than the
violation logic.

So read the output in this order and stop at the first failure: was the
vehicle detected at all (class-id table), did its track id survive ~18
consecutive frames (the dump), did the anchor dot land on the road
(annotated video), and only then, did any zone become trusted. Only the
last one is a statement about the logic.

In [ ]:
!pip install -q yt-dlp

import os
import subprocess

import yt_dlp

URL = "https://youtu.be/Gn-7qyyZ8IY"

# Ask for a video-only stream, not "mp4". YouTube caps its pre-merged mp4
# at 360p, so the obvious `{"format": "mp4"}` silently hands back 640x360 --
# and at 360p an approaching vehicle is a few pixels wide until it is almost
# on top of you, which is far too late for a module that needs ~18
# consecutive frames on one track id. This clip has a 1280x720 stream; take
# it. Audio is irrelevant here, so no merge and no ffmpeg dependency.
FORMAT = "bv*[height<=720][ext=mp4]/bv*[ext=mp4]/mp4"

# The dashcam segment inside the package, established by stepping through
# the frames rather than guessed. This 73-second news piece contains two
# dashcam passes of the same event -- roughly 0-15 s and 29-38 s -- with
# interviews, crash-scene b-roll and a reporter standup filling the rest.
#
# Take the second pass. The first one has a spotlight vignette drawn over
# it by the broadcaster to point the vehicle out to viewers, which darkens
# everything outside a circle and is not something RF-DETR should be asked
# to see through. The second pass is the clean footage.
START = "00:00:33"
DURATION = "00:00:05"

SOURCE = "/content/youtube_source.mp4"
TRIMMED = "/content/youtube_dashcam.mp4"

with yt_dlp.YoutubeDL({"format": FORMAT, "outtmpl": SOURCE, "quiet": True}) as ydl:
    ydl.download([URL])

# Re-encode rather than stream-copy. `-c copy` can only cut on a keyframe,
# so it silently snaps START to somewhere up to a couple of seconds away --
# which on a segment this short is the difference between the dashcam pass
# and the interview beside it.
subprocess.run(
    ["ffmpeg", "-loglevel", "error", "-ss", START, "-t", DURATION,
     "-i", SOURCE, "-c:v", "libx264", "-an", "-y", TRIMMED],
    check=True,
)

# Check first, assign second -- same reason as 3b. A failed trim leaving a
# stale file here would otherwise replace a working VIDEO, and the failure
# would surface much later, in the run cell.
assert os.path.isfile(TRIMMED) and os.path.getsize(TRIMMED) > 0, (
    "Trim produced nothing. Check that START is inside the video's length."
)

# Report what the trim actually produced. Resolution matters downstream:
# `zone_size` is an absolute pixel count, so the same value covers a
# different share of the road at every resolution. If this prints 360, the
# format selector fell through -- fix that before reading any result.
import cv2

probe = cv2.VideoCapture(TRIMMED)
print("frames    :", int(probe.get(cv2.CAP_PROP_FRAME_COUNT)))
print("fps       :", probe.get(cv2.CAP_PROP_FPS))
print("resolution: {0}x{1}".format(
    int(probe.get(cv2.CAP_PROP_FRAME_WIDTH)), int(probe.get(cv2.CAP_PROP_FRAME_HEIGHT))))
probe.release()

VIDEO = TRIMMED
print("Video ready:", VIDEO)

## 4. Run

`limit_frames=300` keeps the first run short: long enough to produce the
class-id table and show whether tracking holds, short enough that a
misconfiguration costs seconds rather than an hour. Drop it once the
class ids are confirmed.

In [ ]:
from road_crime.pipeline import run

# One name, defined once and reused by the download cell. Hardcoding the
# filename in two places is how you end up downloading a stale dump from
# an earlier run and debugging output the code never produced.
TRACKS = "tracks.jsonl"

alerts = run(
    video=VIDEO,
    output="check.mp4",
    limit_frames=300,
    # Records what the violation modules actually consume. See section 7 --
    # this is what makes logic debugging fast.
    dump_tracks=TRACKS,
)
alerts

### 4b. What the logic actually decided — and where it stopped deciding

`alerts` above is a count, and a count cannot tell the six causes of zero
apart. The vehicle may never have had enough history for a heading, never
cleared the speed gate, never stood in a zone with a learned baseline —
or it may have been judged and found to be driving perfectly normally.
Those are entirely different results, and only the last two are the
decision logic saying anything at all.

`diagnose.py` replays the dump the run just wrote and reports, per track,
which gate each frame died at, what baseline each zone learned, and how
each vehicle's apparent size changed. It reads the detector's own
methods — `_heading`, `required_speed`, `baseline.state`,
`peer_consensus` — in the order `update` consults them, so the table
cannot drift away from what the module actually did. Same reason
`replay._trace` reads state rather than recomputing it.

This is what to read when the run says zero. It runs on the dump, so it
costs no GPU and works identically on your laptop:

```
python -m road_crime.diagnose tracks.jsonl
python -m road_crime.diagnose tracks.jsonl --min-speed-fraction 0.01
```

Measured on real head-on footage, every one of the violator's 134 frames
died at `too slow` or `untrusted` and not one reached a comparison — so
the zero said nothing about the wrong-way rule, which was never
consulted. Without this table that run looks like the logic deciding the
driver was innocent.

In [ ]:
from road_crime.diagnose import report

summary = report(TRACKS)

## 5. Watch the annotated result

Green box = tracked vehicle, red = alerted, orange dot = the
road-contact point the detector actually reasons about. If those dots are
not landing on the road beneath each vehicle, fix that before tuning any
threshold -- everything downstream depends on that point being right.

OpenCV writes `mp4v`, which the notebook player will not decode, so
re-encode to H.264 first. The video is inlined as base64, which is fine
for a few hundred frames; for a full clip, download it instead.

In [ ]:
from base64 import b64encode

from IPython.display import HTML

!ffmpeg -loglevel error -i check.mp4 -vcodec libx264 -y check_h264.mp4

payload = b64encode(open("check_h264.mp4", "rb").read()).decode()
HTML('<video width=720 controls><source src="data:video/mp4;base64,{0}">'.format(payload))

In [ ]:
# Longer clips: copy the result back to Drive instead of inlining it.
!cp check_h264.mp4 /content/drive/MyDrive/dashcam/

## 6. Tuning

Once the class ids are confirmed, the next open task is tuning
`DetectorConfig` against real footage. Pass one in explicitly rather than
editing the module, so the tested defaults stay intact:

```python
from road_crime.wrong_way_detector import DetectorConfig

alerts = run(
    video=VIDEO,
    output="check.mp4",
    config=DetectorConfig(opposite_cos_threshold=-0.6),
)
```

Per `CLAUDE.md` principle 3, tune toward silence. A false positive
accuses an innocent driver; a false negative merely misses one.

## 7. Take the track data home — stop iterating through the GPU

The run cell wrote `tracks.jsonl`: per frame, every track's id and
road-contact point. That is the entire input surface a violation module
has (`CLAUDE.md` principle 5), so **everything downstream of perception
can be reproduced from it exactly** — with no GPU, no model, no video and
no third-party packages.

This matters because of arithmetic. Iterating on the logic through this
notebook costs minutes per attempt: edit, push, pull, reload the model,
decode video. Replaying the same run locally costs under a second. When
the thing being debugged is the logic rather than the perception — which
is nearly always — there is no reason to pay the GPU cost again.

Download the file, drop it in the project folder, and work locally:

```
python -m road_crime.replay tracks.jsonl
python -m road_crime.replay tracks.jsonl --track 16 --verbose
python -m road_crime.replay tracks.jsonl --zone-size 240 --opposite-cos-threshold -0.6
```

`--track N --verbose` prints that one vehicle's per-frame decision trail —
heading, zone, whether the zone was trusted, cosine, streak. It is the
fastest way to see why a specific alert fired. The threshold flags let a
parameter sweep be a shell loop instead of a code edit.

In [ ]:
import glob
import os

from google.colab import files

# Prefer the name the run cell used; fall back to whatever dump is newest.
# Never assume a file is there -- an empty runtime and a stale file from an
# earlier run look identical from here, and the second one is worse,
# because you end up analysing output the current code never produced.
target = globals().get("TRACKS")
if not target or not os.path.isfile(target):
    dumps = sorted(glob.glob("*.jsonl"), key=os.path.getmtime, reverse=True)
    print("dumps present:", dumps or "none")
    target = dumps[0] if dumps else None

if target is None:
    print("No dump here yet. Run the pipeline cell first.")
else:
    print("downloading {0} ({1} KB, written {2})".format(
        target,
        os.path.getsize(target) // 1024,
        __import__("time").ctime(os.path.getmtime(target)),
    ))
    files.download(target)

## 8. Build a corpus without downloading anything by hand

Everything above processes one clip that somebody put there. That does not
scale to the twenty-odd clips threshold tuning needs, and nobody should be
downloading videos to a laptop to re-upload them.

`corpus.py` names a source and the clips arrive:

```python
from road_crime.corpus import build_dumps, fetch

clips = fetch("hf:smart-dashcam/motorcycle-accident-driving-datasets", limit=10)
build_dumps(clips, out_dir="fixtures", limit_frames=600)
```

Sources are `url:`, `hf:`, `kaggle:`, or a path. Hugging Face public
repositories need no credentials; Kaggle needs an API token once per
machine. Loose video files are preferred, and archives are searched
otherwise — most driving datasets ship WebDataset tarballs rather than
loose clips, so a fetcher that only understands loose files finds nothing
in them.

**What comes back is dumps, not video.** A clip is ~20 MB and needs a GPU;
its dump is ~200 KB, needs nothing, and is the whole of what a violation
module ever sees. So video is fetched once on a machine that does not care
— this one — and only the dumps travel home, into `fixtures/`, where they
replay for free forever and reach your partner through `git pull`.

**None of this needs labelling.** Ordinary driving footage contains no
wrong-way driving to any useful approximation, so every alert raised on it
is one that should not have been raised — a false-positive rate measured
straight from unlabelled video. Positives come from turning real
trajectories around instead, which is the injection pass inside
`evaluate.py` (`_injection_caught`) — there is no separate `inject.py`.

In [ ]:
!pip install -q huggingface_hub kagglehub

import sys

for _m in [n for n in list(sys.modules) if n.startswith("road_crime")]:
    sys.modules.pop(_m, None)
from road_crime.corpus import build_dumps, fetch

# Pick a source. Start small: ten clips is enough to see whether the corpus
# route works before spending an hour on a hundred.
SOURCE = "hf:smart-dashcam/motorcycle-accident-driving-datasets"

clips = fetch(SOURCE, limit=10)
print("\n{0} clip(s) fetched\n".format(len(clips)))

# limit_frames keeps a first corpus run to minutes rather than an hour.
# Drop it once the route is proven.
build_dumps(clips, out_dir="fixtures", limit_frames=600)

In [ ]:
import glob
import os
import shutil

from google.colab import files

# Bring the whole corpus home in one archive. Dumps are small enough that
# a hundred of them still fit comfortably in a repository.
dumps = sorted(glob.glob("fixtures/*.jsonl"))
total = sum(os.path.getsize(path) for path in dumps)
print("{0} dump(s), {1} KB total".format(len(dumps), total // 1024))

if dumps:
    shutil.make_archive("corpus", "zip", "fixtures")
    print("corpus.zip:", os.path.getsize("corpus.zip") // 1024, "KB")
    files.download("corpus.zip")